In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from data.pre_process import parse_hhmmss_seconds, parse_mmss_seconds

## Objectif

Ici nous nous proposons d'explorer la base de données initiale afin d'indentifier 
quelles sont les variables pertinentes à jeter, c'est à dire les variables qui n'ont
aucune valeur dans le sens où elles n'apportent aucune information, ou les variables qui
sont rendondantes, c'est à dire qu'elles affichent exactement la même données qu'une autre 
colonne mais sous une autre forme, par exemple parse qui est l'inverse de la vitesse.

Nous verrons ensuite comment filtrer les lignes, on verra que cela va surtout se 
concentrer sur les données des splits qui sont parfois lacunaire, ou même désordonnées.

Et nous appliquerons ensuit ces modification sur donnees_finale.parquet à travers le 
fichier pre_process.data


In [ ]:
DBO = r"data/test_splits_clean.csv"

In [ ]:
dfo = pl.read_csv(DBO)

In [ ]:

time_cols = [c for c in dfo.columns if c.endswith("Time")]


pace_cols = [c for c in dfo.columns if c.endswith("pace")]

dfc = dfo.with_columns(
    [parse_hhmmss_seconds(c) for c in time_cols] +
    [parse_mmss_seconds(c) for c in pace_cols]
)

In [ ]:
dfa = dfc.filter(
    [pl.col(f"split_{n}_position") == n+1 for n in range(1, 11)]
)

Ici nous avons créée une base de données où les splits sont bien organisé, ce qui donne
un sens à cette catégorie de variable, en plus il y a seulement 2000 individu sur 57000
environ où les splits ne sont pas organisés

In [ ]:
dfa

In [ ]:
dfc

In [ ]:
dfo

## 1. Id et Race ID

In [ ]:
dfo["bib"].n_unique() == dfo.shape[0]

In [ ]:
dfo["raceId"].unique()

On voit ici que raceId est unique et que bib peut définir une id étant donné sa 
bijectivité

## 2. Split n

In [ ]:
abondonne = [pl.col(f"split_{n}_distance") == 42.195 for n in range(1, 11)]
f"{dfc.filter(pl.any_horizontal(abondonne)).shape[0]} personnes n'ont pas abondonné, soit la totalité"

In [ ]:
with pl.Config(tbl_rows=100):
    print(dfc["split_2_location"].unique())

On voit ici que le split 2 contient beaucoup d'autre splits car pas toute les splits ne
sont enregistré chez les jours

In [ ]:
with pl.Config(tbl_cols=150):
    resultat = dfc.filter(
        (pl.col("split_5_distance") != 42.195) &
        (pl.col("split_6_distance").is_null())
        )

    if resultat.shape[0] > 0:
        print(resultat[0])
    else:
        print("Aucune ligne ne correspond à ces critères !")

Ici on voit par exemple que plusieur splits manquent, et en plus les 2 derniers split
ne sont pas à la bonne place

In [ ]:
dfc.select(pl.all_horizontal(
    [pl.col(f"split_{n}_distance").is_not_null() for n in range(1, 11)]
).sum()).item()

In [ ]:
dfa.shape[0]

On voit ici que seulement 5 individu ont tout leurs split remplis mais ne sont pas bien
trié, ce qui est peu, donc on se permet aussi de filtrer ceux pour qui les splits ne sont 
pas bien organisé

In [ ]:
mapping = {
    1 : ("KM5", 5.0),
    2 : ("KM10", 10.0),
    3 : ("KM15", 15.0),
    4 : ("KM20", 20.0),
    5 : ("KM21", 21.1),
    6 : ("KM25", 25.0),
    7 : ("KM30", 30.0),
    8 : ("KM35", 35.0),
    9 : ("KM40", 40.0),
    10 : ("FINISH", 42.195)
}

cond = (
    [pl.col(f"split_{n}_location") == mapping[n][0] for n in range(1, 11)] +
    [pl.col(f"split_{n}_distance") == mapping[n][1] for n in range(1, 11)] +
    [pl.col(f"split_{n}_position") == n+1 for n in range(1, 11)]
)

In [ ]:
dfc.filter(cond).shape[0]

Ici on voit que location, distance, position donne les mêmes information donc il faut en
garder un et enlever les autres, on garde distance.

## 3.1 realTime vs OfficialTime

L'official time est le temps écoulé après que le tout premier coureur soit partie, 
et real time est le temps écoulé du coureur, donc c'est le vrai temps personnelle.
Official time est inutile à première vu mais il permet de modéliser le "retard", c'est à
l'heure à laquelle le joueur a commencé.

In [ ]:
df_ro = dfc.filter(
    pl.col("officialTime").is_not_null() &
    pl.col("realTime").is_not_null()
).select(["officialTime", "realTime"])

In [ ]:
df_ro = df_ro.with_columns(
    (pl.col("officialTime") - pl.col("realTime")).alias("diff")
)
with pl.Config(tbl_rows=20):
    print(df_ro.sort("diff", descending=True).head(20))

In [ ]:
df_ro["diff"].mean()

## 3.2 realTime vs OfficialTime n split

In [ ]:
column_ros = [col for pair in zip(
    [f"split_{n}_realTime" for n in range(1, 11)],
    [f"split_{n}_officialTime" for n in range(1, 11)]
) for col in pair]

df_ros = dfa.select(column_ros)

df_ros = df_ros.with_columns(
    [(
        pl.col(f"split_{n}_officialTime") - pl.col(f"split_{n}_realTime")
        ).alias(f"split_{n}_diff") for n in range(1, 11)]
)

In [ ]:
df_ros

In [ ]:
df_max = df_ros.select(
    pl.col("^split_.*_diff$").mean()
)

df_max

On remarque que diff sur split1 est assez proche du retard définie précédemment, c'est
enfaite car il garde la même logique, temps écoulé entre le passe en split1 et le départ
du 1er coureur. alors que les autres sont assez petit donc officialtime ne sert pratiquement 
à rien 

In [ ]:
with pl.Config(tbl_cols=150):
    print(dfa[df_ros.select(pl.col("split_2_diff").arg_max()).item()])

On remarque deplus ici que lorsqu'on a de grands écart sur la diff Time sur split,
c'est probablement une erreur machine car realTime est à 0 est speed est sur null.
On préfère supprimer ces lignes.